# Gnani Prisma (ASR) & Timbre (TTS) Evaluation

**Engineer:** Ashish Vipin Korumelli  
**Role:** AI Solutions Engineer Evaluation  
**Target Systems:** Gnani Prisma v3.0 (ASR) & Timbre v2.5 (TTS) via api.vachana.ai

---

## Overview
This notebook executes an end-to-end benchmarking harness evaluating:
1. **Prisma ASR**: 30 test utterances spanning 11 categories (Clean, Noisy, Telephony, Numbers/ITN, Code-Switching, Accents, Domain-Specific, Length variations, and Regional Indic languages).
2. **Timbre TTS**: 28 test sentences spanning 9 categories evaluated across Latency, RTF, Speaking Rate, Silence Ratio, Clipping, and Round-Trip ASR transcription fidelity.
3. **Metric Implementation**: WER & CER calculated from scratch via Wagner-Fischer dynamic programming algorithm.

In [ ]:
import os
import sys
import json
from pathlib import Path

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))

from config import GNANI_API_KEY, ASR_MODEL, TTS_MODEL
from src.metrics import compute_wer, compute_cer, aggregate_wer, aggregate_cer
from src.gnani_client import GnaniClient

print('API Key Configured:', 'Yes' if GNANI_API_KEY else 'No')
print(f'Target Models: ASR={ASR_MODEL}, TTS={TTS_MODEL}')

## Part 1: Metric Verification (Scratch Levenshtein DP)
Verifying our custom Levenshtein implementation with substitution (S), deletion (D), and insertion (I) breakdown.

In [ ]:
sample_ref = 'the cat sat on the mat'
sample_hyp = 'the cat sit on a mat'

res_wer = compute_wer(sample_ref, sample_hyp)
res_cer = compute_cer(sample_ref, sample_hyp)

print('Reference :', sample_ref)
print('Hypothesis:', sample_hyp)
print(f"WER: {res_wer['wer']:.2%} (S={res_wer['substitutions']}, D={res_wer['deletions']}, I={res_wer['insertions']})")
print(f"CER: {res_cer['cer']:.2%}")

## Part 2: Prisma ASR Empirical Benchmark Results
Loading and reviewing the evaluation results generated against Prisma v3.0.

In [ ]:
with open('results/asr_results.json', 'r', encoding='utf-8') as f:
    asr_res = json.load(f)

print('=' * 50)
print(f"PRISMA ASR OVERALL BENCHMARK (n={asr_res['metadata']['total_clips']})")
print(f"Overall Micro-averaged WER: {asr_res['overall']['wer']:.2%}")
print(f"Overall Micro-averaged CER: {asr_res['overall']['cer']:.2%}")
print('=' * 50)

header = f"{'Category':<26} | {'WER':<8} | {'CER':<8} | {'Avg Latency':<12} | {'Clips'}"
print(header)
print('-' * len(header))
for cat, data in asr_res['by_category'].items():
    print(f"{cat:<26} | {data['aggregate_wer']*100:>6.2f}% | {data['aggregate_cer']*100:>6.2f}% | {data['avg_latency_ms']:>8.1f} ms | {data['num_clips']:>5}")

### Qualitative Error Analysis: The Inverse Text Normalisation (ITN) Trade-off
Inspect how Prisma normalises numbers, entities, and code-mixed terms into production-ready representations.

In [ ]:
for r in asr_res['per_clip_results']:
    if r['category'] in ['numbers_and_entities', 'code_switching', 'telephony']:
        print(f"ID: {r['id']} ({r['category']})")
        print(f"  Ref: {r['reference']}")
        print(f"  Hyp: {r['hypothesis']}")
        print(f"  WER: {r['wer']*100:.1f}% | CER: {r['cer']*100:.1f}%")
        print('-' * 40)

## Part 3: Timbre TTS Quality & Latency Benchmark Results
Inspecting the synthesis latency, Real-Time Factor (RTF), speaking rate, silence ratio, and automated quality scoring.

In [ ]:
with open('results/tts_results.json', 'r', encoding='utf-8') as f:
    tts_res = json.load(f)

print('=' * 50)
print(f"TIMBRE TTS OVERALL BENCHMARK (n={tts_res['metadata']['total_sentences']})")
print(f"Average Quality Score: {tts_res['overall']['avg_quality_score']:.1f}/100")
print(f"Average Real-Time Factor (RTF): {tts_res['overall']['avg_rtf']:.4f}")
print(f"Average Latency: {tts_res['overall']['avg_latency_ms']:.1f} ms")
print(f"Latency Range: {tts_res['overall']['min_latency_ms']:.1f} ms - {tts_res['overall']['max_latency_ms']:.1f} ms")
print('=' * 50)

header = f"{'Category':<26} | {'Score':<8} | {'RTF':<8} | {'Avg Latency':<12} | {'WPM':<6}"
print(header)
print('-' * len(header))
for cat, data in tts_res['by_category'].items():
    print(f"{cat:<26} | {data['avg_quality_score']:>6.1f} | {data['avg_rtf']:>8.4f} | {data['avg_latency_ms']:>8.1f} ms | {data['avg_speaking_rate_wpm']:>6.1f}")

## Part 4: Interactive Test Harness
Call Prisma ASR or Timbre TTS on any custom utterance or sentence on-the-fly.

In [ ]:
client = GnaniClient()

# Test TTS generation
test_text = 'नमस्ते, सनराइज़ बैंक में आपका स्वागत है। आपका अकाउंट बैलेंस 50,000 रुपये है।'
output_audio = 'data/tts/output/demo_sample.wav'

tts_out = client.synthesize(text=test_text, language='hi-IN', output_path=output_audio)
print(f"Synthesized in {tts_out['latency_ms']} ms -> {output_audio}")

# Transcribe it right back with Prisma ASR
asr_out = client.transcribe(output_audio, language='hi-IN')
print(f"Transcribed in {asr_out['latency_ms']} ms:")
print(f"Output: {asr_out['transcript']}")